# Can Learning-to-Rank Prioritize Content Decline Reviews Better Than a Fixed Rule?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalalostcode/FlyrankAI_ML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

## Abstract

This study asks whether a learned ranking model can prioritize content items showing a current decline signal more precisely than a fixed editorial rule. I used a public-safe FlyRank starter dataset containing 30,000 pseudonymized content items from 32 clients, with trailing-90-day search, engagement, and content metadata. I compared a transparent weighted rule with an equal fold-rank blend of two LightGBM LambdaRank models under five-fold cross-validation grouped by client, while excluding label-derived fields, outcome-window fields, identifiers, and product scores from the model matrix. Across the same held-out folds, the selected blend measured **92.4% ± 3.3% mean precision@50**, compared with **46.4% ± 15.7%** for the rule and a **54.4%** mean fold base rate; its mean ROC-AUC was a more modest **0.643**, so the result is specific to top-of-queue ranking. The output is a decision-support queue that tells an editor what to inspect first, not a future forecast, causal claim, or instruction to edit automatically.

## 1. Question

**Research question:** Can a client-grouped learning-to-rank model prioritize current content-decline reviews more precisely than a transparent fixed rule at the top 50 slots?

The unit is one pseudonymized content item. The decision is which items a human editor should inspect first when review capacity is limited. False positives consume review time and can prompt unnecessary changes; false negatives delay investigation.

In [1]:
from pathlib import Path
import hashlib
import importlib.util
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

def find_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'skills' / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the FlyRank repository.')

ROOT = find_repo_root()
OUT = ROOT / 'work' / 'outputs'
REPORT_PATH = ROOT / 'work' / 'capstone_report.md'

def read_json(path):
    with path.open(encoding='utf-8') as handle:
        return json.load(handle)

v1_results = read_json(OUT / 'feature_engineering_results.json')
v2_results = read_json(OUT / 'feature_engineering_v2_results.json')
advanced = read_json(OUT / 'advanced_ranking_results.json')
action = read_json(OUT / 'action_playbook_metrics.json')
stacking = read_json(OUT / 'stacking_probe_results.json')
calibration = read_json(OUT / 'hierarchical_calibration_probe_results.json')

baseline = v1_results['results']['R0_rule_baseline']['summary']
selected = advanced['results']['B2_current_rank_blend']['summary']
selected_folds = advanced['results']['B2_current_rank_blend']['folds']
baseline_folds = v1_results['results']['R0_rule_baseline']['folds']

research_question = (
    'Can a client-grouped learning-to-rank model prioritize current content-decline '
    'reviews more precisely than a transparent fixed rule at the top 50 slots?'
)
print(research_question)
print('Decision: which pseudonymous content items should a human editor inspect first?')
print(f'Primary metric: {advanced["primary_metric"]}')
print(f'Random seed: {RANDOM_STATE}')

Can a client-grouped learning-to-rank model prioritize current content-decline reviews more precisely than a transparent fixed rule at the top 50 slots?
Decision: which pseudonymous content items should a human editor inspect first?
Primary metric: mean precision@50 across the same five fixed GroupKFold folds
Random seed: 42


## 2. Data

The starter release contains 30,000 rows × 44 columns, one pseudonymized content item per row, across 32 client groups. The proxy target is 1 when \`trend_direction == "down"\` (a change below −20% between the latest and previous 30-day impression windows), and 0 otherwise.

No client names, domains, URLs, titles, keywords, or raw queries are used. Label sources, latest-window siblings, identifiers, and product scores are excluded from features. Rates use a 0–100 percentage scale; \`avg_position = 0\` is a missing-rank sentinel.

In [2]:
raw_path = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
raw = pd.read_csv(raw_path)

assert raw.shape == (30_000, 44)
assert raw['client_id'].nunique() == 32
assert raw['content_id'].is_unique
assert int((raw['trend_direction'] == 'down').sum()) == 16_262
assert int(raw['avg_position'].eq(0).sum()) == 1_205

positive_rate = float((raw['trend_direction'] == 'down').mean())
print(f'Rows × columns: {raw.shape[0]:,} × {raw.shape[1]}')
print(f'Pseudonymous client groups: {raw["client_id"].nunique()}')
print(f'Decline-proxy positive rate: {positive_rate:.2%}')
print(f'avg_position=0 (missing-rank sentinel): {raw["avg_position"].eq(0).sum():,}')
print('Public-safety check: aggregate output only; no identifiers, URLs, or queries printed.')

forbidden_model_fields = {
    'trend_direction', 'trend_pct', 'is_declining_label',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'content_id', 'client_id',
}
print(f'Forbidden target/window/ID fields documented: {len(forbidden_model_fields)}')

Rows × columns: 30,000 × 44
Pseudonymous client groups: 32
Decline-proxy positive rate: 54.21%
avg_position=0 (missing-rank sentinel): 1,205
Public-safety check: aggregate output only; no identifiers, URLs, or queries printed.
Forbidden target/window/ID fields documented: 8


## 3. Methodology

The transparent baseline weights visibility (40%), freshness risk (30%), position opportunity (25%), and content-depth gap (5%). The selected \`B2_current_rank_blend\` is a fixed 50/50 fold-percentile blend of base-feature and engineered-feature LightGBM LambdaRank models.

All comparisons use the same five \`GroupKFold(client_id)\` splits and out-of-fold scores. The workflow asserts forbidden fields are absent, checks row alignment, fixes seed 42, and includes a deliberate leakage confession test. A previous-window feature family was rejected when it partially reconstructed the snapshot label.

In [3]:
module_path = ROOT / 'work' / 'scripts' / 'feature_engineering_experiment.py'
spec = importlib.util.spec_from_file_location('feature_engineering_v1', module_path)
if spec is None or spec.loader is None:
    raise ImportError(f'Cannot load {module_path}')
v1_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v1_module)

requested_features = (
    set(v1_module.BASE_NUMERIC_FEATURES)
    | set(v1_module.BASE_CATEGORICAL_FEATURES)
    | set(v1_module.ENGINEERED_NUMERIC_FEATURES)
)
assert requested_features.isdisjoint(v1_module.FORBIDDEN_MODEL_COLUMNS)
assert advanced['feature_counts'] == {'base': 52, 'engineered': 80}

blend = v2_results['results']['F7_lambda_pair_equal_blend']
assert blend['components'] == [
    'B0_engineered_lambdarank',
    'B1_base_lambdarank',
]
assert blend['weights'] == [0.5, 0.5]

print(f'Base source features: {len(v1_module.BASE_NUMERIC_FEATURES)} numeric + '
      f'{len(v1_module.BASE_CATEGORICAL_FEATURES)} categorical')
print(f'Engineered numeric additions: {len(v1_module.ENGINEERED_NUMERIC_FEATURES)}')
print(f'Encoded matrices: {advanced["feature_counts"]["base"]} base columns; '
      f'{advanced["feature_counts"]["engineered"]} engineered columns')
print('Selected model: equal fold-rank blend of base and engineered LambdaRank.')
print('Forbidden-column intersection: empty (PASS)')
print('Validation: 5 GroupKFold splits by client_id; every row receives one OOF score.')

Base source features: 18 numeric + 8 categorical
Engineered numeric additions: 28
Encoded matrices: 52 base columns; 80 engineered columns
Selected model: equal fold-rank blend of base and engineered LambdaRank.
Forbidden-column intersection: empty (PASS)
Validation: 5 GroupKFold splits by client_id; every row receives one OOF score.


## 4. Results (vs baseline)

The selected blend measured **92.4% ± 3.3% mean P@50**, versus **46.4% ± 15.7%** for the rule and a **54.4%** mean fold base rate on the same five held-out-client folds. This is a +46.0 percentage-point difference from the rule. Mean ROC-AUC was 0.643 and average precision was 0.684, so the model is described as a top-of-queue specialist, not a universal probability model.

In [4]:
comparison = pd.DataFrame([
    {
        'method': 'Mean fold base rate',
        'P@20': selected['mean_base_rate'],
        'P@50': selected['mean_base_rate'],
        'P@50 SD': selected['std_base_rate'],
        'P@100': selected['mean_base_rate'],
        'ROC-AUC': 0.5,
        'Average precision': selected['mean_base_rate'],
    },
    {
        'method': 'Transparent rule',
        'P@20': baseline['mean_p_at_20'],
        'P@50': baseline['mean_p_at_50'],
        'P@50 SD': baseline['std_p_at_50'],
        'P@100': baseline['mean_p_at_100'],
        'ROC-AUC': baseline['mean_roc_auc'],
        'Average precision': baseline['mean_average_precision'],
    },
    {
        'method': 'Selected rank blend',
        'P@20': selected['mean_p_at_20'],
        'P@50': selected['mean_p_at_50'],
        'P@50 SD': selected['std_p_at_50'],
        'P@100': selected['mean_p_at_100'],
        'ROC-AUC': selected['mean_roc_auc'],
        'Average precision': selected['mean_average_precision'],
    },
])
formatted = comparison.copy()
for column in ['P@20', 'P@50', 'P@50 SD', 'P@100']:
    formatted[column] = formatted[column].map(lambda value: f'{value:.1%}')
for column in ['ROC-AUC', 'Average precision']:
    formatted[column] = formatted[column].map(lambda value: f'{value:.3f}')
print(formatted.to_string(index=False))

p50_gain_rule_pp = 100 * (selected['mean_p_at_50'] - baseline['mean_p_at_50'])
p50_gain_base_pp = 100 * (selected['mean_p_at_50'] - selected['mean_base_rate'])
assert round(p50_gain_rule_pp, 1) == 46.0
assert round(p50_gain_base_pp, 1) == 38.0

figure_path = OUT / 'paper_results_summary.png'
fig, axes = plt.subplots(1, 2, figsize=(13, 5.3))

ks = ['P@20', 'P@50', 'P@100']
x = np.arange(len(ks))
width = 0.24
series = [
    ('Base rate', [selected['mean_base_rate']] * 3, '#999999'),
    ('Rule', [baseline['mean_p_at_20'], baseline['mean_p_at_50'], baseline['mean_p_at_100']], '#e68613'),
    ('Selected blend', [selected['mean_p_at_20'], selected['mean_p_at_50'], selected['mean_p_at_100']], '#386cb0'),
]
for offset, (label, values, color) in enumerate(series):
    bars = axes[0].bar(x + (offset - 1) * width, values, width, label=label, color=color)
    axes[0].bar_label(bars, labels=[f'{value:.0%}' for value in values], padding=3, fontsize=9)
axes[0].set_xticks(x, ks)
axes[0].set_ylim(0, 1.08)
axes[0].set_ylabel('Precision / rate')
axes[0].set_title('Same folds: top-K precision')
axes[0].legend(frameon=False, loc='lower right')

fold_ids = [row['fold'] for row in selected_folds]
axes[1].plot(
    fold_ids, [row['p_at_50'] for row in baseline_folds],
    marker='o', linewidth=2, label='Rule P@50', color='#e68613'
)
axes[1].plot(
    fold_ids, [row['p_at_50'] for row in selected_folds],
    marker='o', linewidth=2, label='Selected blend P@50', color='#386cb0'
)
axes[1].plot(
    fold_ids, [row['base_rate'] for row in selected_folds],
    marker='o', linewidth=2, linestyle='--', label='Fold base rate', color='#777777'
)
axes[1].set_xticks(fold_ids)
axes[1].set_ylim(0, 1.08)
axes[1].set_xlabel('Held-out client fold')
axes[1].set_ylabel('Precision / rate')
axes[1].set_title('P@50 varies with held-out clients')
axes[1].legend(frameon=False, loc='lower right')

fig.suptitle('Client-grouped evaluation of the content review queue', fontsize=15, fontweight='bold')
fig.tight_layout()
fig.savefig(figure_path, dpi=170, bbox_inches='tight')
plt.close(fig)

print(f'\nP@50 difference vs rule: {p50_gain_rule_pp:+.1f}pp')
print(f'P@50 difference vs mean fold base rate: {p50_gain_base_pp:+.1f}pp')
print(f'Mean fold lift@50: {selected["mean_lift_at_50"]:.2f}×')
print(f'Figure written: {figure_path.relative_to(ROOT)}')

             method  P@20  P@50 P@50 SD P@100 ROC-AUC Average precision
Mean fold base rate 54.4% 54.4%   11.0% 54.4%   0.500             0.544
   Transparent rule 47.0% 46.4%   15.7% 46.4%   0.600             0.596
Selected rank blend 92.0% 92.4%    3.3% 88.6%   0.643             0.684

P@50 difference vs rule: +46.0pp
P@50 difference vs mean fold base rate: +38.0pp
Mean fold lift@50: 1.76×
Figure written: work\outputs\paper_results_summary.png


## 5. Limitations

This is a single-snapshot, cross-client ranking study. Inherited 90-day predictors overlap the current label period, the target is a thresholded proxy rather than ground truth, and no temporal holdout or randomized intervention was run. The rank score is not a probability and cannot establish why an item declined or whether an edit will improve traffic.

The advanced 92.8% candidate gained only +0.4pp on the same development folds, below the predeclared +5pp target, so it remains in shadow status.

In [5]:
shadow = advanced['results']['A11_current_plus_top20_equal']['summary']
assert advanced['target_achieved'] is False
assert round(advanced['improvement_pp'], 1) == 0.4
assert round(stacking['winner_mean_p_at_50'], 3) == 0.896
assert round(calibration['improvement_pp'], 1) == 0.0

safe_claim = (
    f'On five client-grouped folds in this single-snapshot dataset, the selected '
    f'rank blend measured {selected["mean_p_at_50"]:.1%} ± '
    f'{selected["std_p_at_50"]:.1%} P@50 versus '
    f'{baseline["mean_p_at_50"]:.1%} ± {baseline["std_p_at_50"]:.1%} '
    f'for the fixed rule. This supports current-snapshot review prioritization, '
    f'not future forecasting or causal refresh impact.'
)
banned = ['proves', 'causes', 'will increase', 'guarantees', 'predicted google']
assert not any(term in safe_claim.lower() for term in banned)

print(safe_claim)
print('\nLimits verified:')
print('- single snapshot; inherited 90-day predictors overlap the current label period')
print('- threshold proxy at a 20% impression drop, not content-quality ground truth')
print('- no temporal holdout and no randomized intervention')
print('- score is a relative rank, not a calibrated probability')
print(f'- whole-ranking ROC-AUC is moderate: {selected["mean_roc_auc"]:.3f}')
print(f'- best same-fold challenger: {shadow["mean_p_at_50"]:.1%} '
      f'({advanced["improvement_pp"]:+.1f}pp), retained in shadow status')
print(f'- conventional stacking: {stacking["winner_mean_p_at_50"]:.1%}; '
      f'hierarchical calibration gain: {calibration["improvement_pp"]:+.1f}pp')

On five client-grouped folds in this single-snapshot dataset, the selected rank blend measured 92.4% ± 3.3% P@50 versus 46.4% ± 15.7% for the fixed rule. This supports current-snapshot review prioritization, not future forecasting or causal refresh impact.

Limits verified:
- single snapshot; inherited 90-day predictors overlap the current label period
- threshold proxy at a 20% impression drop, not content-quality ground truth
- no temporal holdout and no randomized intervention
- score is a relative rank, not a calibrated probability
- whole-ranking ROC-AUC is moderate: 0.643
- best same-fold challenger: 92.8% (+0.4pp), retained in shadow status
- conventional stacking: 89.6%; hierarchical calibration gain: +0.0pp


## 6. Ranked recommendations

Work inside each client group: ranks 1–10 are \`review_now\`, 11–25 are \`review_next\`, and 26–50 are \`monitor\`. Reason codes choose the inspection—snippet and intent, depth and relevance, freshness, or engagement—but they do not explain the label.

A reviewer records \`act\`, \`defer\`, or \`no_action\` after checking the time series, seasonality, indexing, current intent, factual quality, and business constraints. Never edit, redirect, unpublish, contact a client, evaluate staff, or promise traffic impact from the score alone.

In [6]:
print(f'Queue rows: {action["queue_rows"]:,} across {action["client_groups"]} groups')
print('Priority tiers:')
for tier, count in action['priority_counts'].items():
    print(f'  {tier:<12} {count:>4}')
print('Suggested review actions:')
for name, count in action['action_counts'].items():
    print(f'  {name:<32} {count:>4}')

assert action['selected_model'] == advanced['benchmark']
assert np.isclose(action['monitoring']['p50_pause_floor'], 0.824)
assert set(action['operational_export_excludes']) >= {
    'is_declining_label', 'trend_direction', 'trend_pct', 'fold', 'shadow_rank_score'
}
print('\nPolicy checks:')
print('- review ranks 1–10 first inside each client group')
print('- reason codes are prompts, not explanations')
print('- no automatic edit, redirect, unpublish, outreach, or staff evaluation')
print('- pause after two consecutive mature-label performance alerts')

Queue rows: 1,473 across 32 groups
Priority tiers:
  monitor       695
  review_next   465
  review_now    313
Suggested review actions:
  diagnose_before_action            658
  inspect_search_snippet            521
  review_facts_and_freshness        274
  review_depth_and_relevance         13
  review_intent_and_experience        7

Policy checks:
- review ranks 1–10 first inside each client group
- reason codes are prompts, not explanations
- no automatic edit, redirect, unpublish, outreach, or staff evaluation
- pause after two consecutive mature-label performance alerts


## 7. Artifacts the paper embeds

The manuscript embeds one results chart comparing the base rate, fixed rule, and selected blend across K, plus fold-level P@50. It also embeds the action-playbook composition chart. Aggregate JSON receipts preserve exact fold metrics and the queue policy without publishing row-level identifiers.

In [7]:
report = REPORT_PATH.read_text(encoding='utf-8')
required_sections = [
    '## 0. Abstract',
    '## 1. Problem framing',
    '## 2. Data safety',
    '## 3. Baseline',
    '## 4. Model and methodology',
    '## 5. Evaluation and results',
    '## 6. Interpretation',
    '## 7. Ranked recommendations',
    '## 8. Reproducibility',
    '## 9. Acknowledgments and data credit',
    '### Five-minute demo outline',
    '### Social-post cut',
    '### Employer-facing summary',
]
missing_sections = [section for section in required_sections if section not in report]
assert not missing_sections, missing_sections
assert 'https://flyrank.ai' in report
assert '<your lane>' not in report.lower()
assert 'PASTE-YOUR' not in report
assert 'paper_results_summary.png' in report
assert 'action_playbook_summary.png' in report

artifact_paths = [
    REPORT_PATH,
    OUT / 'paper_results_summary.png',
    OUT / 'action_playbook_summary.png',
    OUT / 'advanced_ranking_results.json',
    OUT / 'action_playbook_metrics.json',
]
for path in artifact_paths:
    assert path.exists() and path.stat().st_size > 0

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

capstone_receipt = {
    'artifact': 'capstone_paper_v1',
    'question': research_question,
    'source_rows': int(raw.shape[0]),
    'client_groups': int(raw['client_id'].nunique()),
    'random_state': RANDOM_STATE,
    'selected_model': advanced['benchmark'],
    'validation': {
        'folds': 5,
        'mean_fold_base_rate': selected['mean_base_rate'],
        'rule_mean_p_at_50': baseline['mean_p_at_50'],
        'rule_std_p_at_50': baseline['std_p_at_50'],
        'model_mean_p_at_50': selected['mean_p_at_50'],
        'model_std_p_at_50': selected['std_p_at_50'],
        'model_mean_roc_auc': selected['mean_roc_auc'],
        'model_mean_average_precision': selected['mean_average_precision'],
        'model_mean_lift_at_50': selected['mean_lift_at_50'],
        'model_fold_p_at_50': [row['p_at_50'] for row in selected_folds],
    },
    'shadow_candidate': {
        'name': advanced['winner'],
        'mean_p_at_50': advanced['winner_mean_p_at_50'],
        'gain_pp': advanced['improvement_pp'],
        'target_achieved': advanced['target_achieved'],
        'status': 'not promoted',
    },
    'queue_rows': action['queue_rows'],
    'required_paper_sections_verified': len(required_sections),
    'artifact_sha256': {
        str(path.relative_to(ROOT)): sha256(path) for path in artifact_paths
    },
}
capstone_receipt_path = OUT / 'capstone_paper_metrics.json'
capstone_receipt_path.write_text(
    json.dumps(capstone_receipt, indent=2),
    encoding='utf-8',
)

reloaded = read_json(capstone_receipt_path)
assert reloaded['validation']['model_mean_p_at_50'] == selected['mean_p_at_50']
assert reloaded['required_paper_sections_verified'] == len(required_sections)
assert (OUT / 'paper_results_summary.png').stat().st_size > 10_000

print(f'Paper sections verified: {len(required_sections)}')
print('Artifacts verified:')
for path in (*artifact_paths, capstone_receipt_path):
    print(f'  {path.relative_to(ROOT)} ({path.stat().st_size:,} bytes)')
print('Aggregate-only capstone receipt written; no row-level identifiers included.')

Paper sections verified: 13
Artifacts verified:
  work\capstone_report.md (18,337 bytes)
  work\outputs\paper_results_summary.png (122,931 bytes)
  work\outputs\action_playbook_summary.png (78,727 bytes)
  work\outputs\advanced_ranking_results.json (37,450 bytes)
  work\outputs\action_playbook_metrics.json (2,234 bytes)
  work\outputs\capstone_paper_metrics.json (1,744 bytes)
Aggregate-only capstone receipt written; no row-level identifiers included.


## 8. Five-minute demo outline

1. **0:00–0:40 — Decision:** limited editorial capacity and why P@50 is primary.
2. **0:40–1:20 — Data and label:** 30,000 items, 32 clients, proxy target, forbidden fields.
3. **1:20–2:10 — Validation:** five client-grouped folds and the rule/base-rate comparison.
4. **2:10–3:10 — Result:** 46.4% rule → 92.4% selected P@50, followed immediately by ROC-AUC 0.643 and limitations.
5. **3:10–4:10 — Playbook:** per-client ranks, reason codes, human review, no-go actions.
6. **4:10–5:00 — Next experiment:** temporal warehouse features, multiple forecast origins, reproducibility receipts.

## 9. Social-post cut

I built a client-grouped learning-to-rank system for prioritizing content-decline reviews on 30,000 pseudonymized items. On the same five held-out-client folds, a transparent rule measured 46.4% mean precision@50, while the selected LambdaRank blend measured 92.4% ± 3.3% (mean fold base rate: 54.4%). The important caveat: this is current-snapshot decision support, not future forecasting or proof that an edit will improve traffic. The repo includes failed experiments, leakage guards, aggregate receipts, and a human-review action playbook.

## 10. Employer-facing summary

I built a reproducible learning-to-rank pipeline and editorial action queue over 30,000 pseudonymized content items from 32 clients. Using five-fold client-grouped validation, the selected model measured 92.4% ± 3.3% precision@50 versus 46.4% ± 15.7% for a transparent rule, while explicit leakage tests and a 0.643 ROC-AUC kept the claim scoped to top-of-queue prioritization. I translated the model into review reason codes, no-go automation rules, drift thresholds, and commit-safe JSON receipts so another analyst can audit and rerun the work.

## Self-check

- [x] Every section above is filled—markdown thinking and code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, data URLs, or private queries appear anywhere
- [x] Claims use observed, measured, directional, and decision-support language
- [x] The paper has Abstract through Acknowledgments/data credit in the required order
- [x] Model, rule, and base rate use the same five client-grouped folds
- [x] Aggregate receipts and figures are reloaded and verified
- [x] ML-12 closing cells contain the demo, social cut, and employer summary
- [ ] Committed to the repo and deployed; then place the direct paper URL in \`submission/paper_url.txt\`